## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/nanotube.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="luis EXPERIMENT: path follower")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

c:\Users\luis\miniforge3\envs\nanover\Lib\site-packages\nglview\__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [9]:
TRAIL_COLOR = [0.0, 1.0, 1.0, 0.5]
PATH_COLOR = [1.0, 1.0, 0.0, 0.5]
HOVER_COLOR = [1.0, 0.0, 0.0, 0.5]
CHOSEN_COLOR = [0.0, 1.0, 0.0, 0.5]

PATH_WIDTH = 0.04
TRAIL_WIDTH = 0.02
HOVER_WIDTH = 0.06
CHOSEN_WIDTH = 0.07

SELECTION_RADIUS = 0.2

In [4]:
from follower import PathFollowerAgent, Path

PATHS: dict[str, Path] = {}
NEXT_PATH_ID = 0
FOLLOWER_AGENT: PathFollowerAgent | None = None


def CREATE_PATH():
    global NEXT_PATH_ID
    path_id = str(NEXT_PATH_ID)
    NEXT_PATH_ID += 1
    PATHS[path_id] = Path.empty()
    return path_id


def REDRAW_PATH(path_id: str):
    path = PATHS[path_id]
    PATH_OBJECTS.update_line(f"path.{path_id}", type="path", positions=path.positions, size=PATH_WIDTH, color=PATH_COLOR)


def STOP_FOLLOWER():
    global FOLLOWER_AGENT
    if FOLLOWER_AGENT is not None:
        FOLLOWER_AGENT.close()
    FOLLOWER_AGENT = None


def START_FOLLOWER(particles: list[int], path: Path):
    global FOLLOWER_AGENT
    STOP_FOLLOWER()
    FOLLOWER_AGENT = PathFollowerAgent.from_runner(imd_runner)
    FOLLOWER_AGENT.set_path(path)
    FOLLOWER_AGENT.set_particles(particles)
    FOLLOWER_AGENT.start()

# Draw path mode

In [5]:
from nanover.jupyter import Mode, SceneObjectsUtility
from follower import Path

PATH_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
CURSOR_DRAWING_PATH_ID = {}


class DrawPathMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_DRAWING_PATH_ID[key] = CREATE_PATH()

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_DRAWING_PATH_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        if not key in CURSOR_DRAWING_PATH_ID:
            return

        path_id = CURSOR_DRAWING_PATH_ID[key]

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        PATHS[path_id].append_position(next_pos)

        REDRAW_PATH(path_id)


utilities.use_interaction_modes()
utilities.add_interaction_mode(DrawPathMode, "draw path", icon="✏️")

# Trail mode

In [6]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode

TRAIL_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
TRAIL_POINTS = {}


def get_interaction_centroid(interaction: ParticleInteraction):
    indexes = [int(index) for index in interaction.particles]
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    centroid = [float(x) for x in np.mean(positions[indexes], axis=0)]
    return centroid


class InteractionTrailMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key] = []

    def on_interaction_updated(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key].append(get_interaction_centroid(interaction))
        TRAIL_OBJECTS.update_line(f"trail.{key}", positions=TRAIL_POINTS[key], size=0.05, color=TRAIL_COLOR)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        utilities.notify_all(f"FINISHED TRAIL\n{key}")

utilities.add_interaction_mode(InteractionTrailMode, "trails", icon="〰️")

# Path following mode

In [7]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from intersection import intersects_sphere_line


SELECTION_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)


def intersect_paths(position, radius):
    for path_id, path in PATHS.items():
        if intersects_sphere_line(position, radius, path.positions):
            return path_id, path
    return None


class PathFollowingMode(Mode):
    PATH_ID = None
    PARTICLES = None

    def check(self):
        if self.PARTICLES is None or self.PATH_ID is None:
            return

        utilities.notify_all(f"FOLLOWING!")
        START_FOLLOWER(self.PARTICLES, PATHS[self.PATH_ID])
        self.PATH_ID = None
        self.PARTICLES = None

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        self.PARTICLES = [int(index) for index in interaction.particles]
        utilities.notify_all(f"SELECTED PARTICLES: {self.PARTICLES}")
        self.check() 

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button != "primary":
            return

        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_paths(next_pos, SELECTION_RADIUS)

        if hovered is not None:
            path_id, path = hovered
            utilities.notify_all(f"SELECTED PATH {path_id}")
            self.PATH_ID = path_id
            SELECTION_OBJECTS.update_line(f"selected.path", positions=path.positions, size=CHOSEN_WIDTH, color=CHOSEN_COLOR)
            self.check()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_paths(next_pos, SELECTION_RADIUS)

        if hovered is None:
            SELECTION_OBJECTS.remove_line(f"hover.{key}")
        else:
            SELECTION_OBJECTS.update_line(f"hover.{key}", positions=hovered[1].positions, size=HOVER_WIDTH, color=HOVER_COLOR)

utilities.add_interaction_mode(PathFollowingMode, "follow", icon="🚂")

# Trail2Path mode

In [8]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode
from traceback import print_exc


def intersect_trails(position, radius):
    for path_id, path in TRAIL_POINTS.items():
        if intersects_sphere_line(position, radius, np.array(path)):
            return path_id, path
    return None


class Trail2PathMode(Mode):

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button != "primary":
            return

        cursor_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_trails(cursor_pos, SELECTION_RADIUS)

        if hovered is not None:
            path_id, path = hovered

            new_path_id = CREATE_PATH()
            print(f"{new_path_id}")
            path = PATHS[new_path_id]
            for point in TRAIL_POINTS[path_id]:
                path.append_position(point)
            REDRAW_PATH(new_path_id)

            TRAIL_POINTS.pop(path_id, None)
            TRAIL_OBJECTS.remove_line(f"trail.{path_id}") # remove  trail

            utilities.notify_all(f"Trail transformed")

    def on_cursor_updated(self, *, key: str, cursor: dict):
        cursor_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_trails(cursor_pos, SELECTION_RADIUS)

        if hovered is None:
            SELECTION_OBJECTS.remove_line(f"hover.{key}")
        else:
            SELECTION_OBJECTS.update_line(f"hover.{key}", positions=hovered[1], size=0.1, color=HOVER_COLOR)

utilities.add_interaction_mode(Trail2PathMode, "trail2path", icon="🛤️")